# turkish_distill: Colab pipeline

Thin notebook. All logic lives in `src/turkish_distill`. Run the cells top to bottom; every step is resumable, so after a disconnect re-run from the setup cell and then the step you were in.

Before the first run: add `HF_TOKEN` to Colab Secrets (needed for gated models) and set `REPO_URL` below.

## 1. Setup (Drive, repo, install)

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

## 2. Choose the experiment

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        print('HF_TOKEN not set (only needed for gated models):', exc)

# --- Experiment choice ---
CONFIG = 'configs/single_teacher_qwen3_8b.yaml'   # or configs/multi_teacher_a.yaml
RUN_ID = None   # set e.g. '20260101-120000-abc1234' to resume an existing run; None starts a new run
OVERRIDES = []  # e.g. ['train.max_steps=200', 'data.max_samples=20000'] for a quick trial

override_args = ' '.join(f'--override {o}' for o in OVERRIDES)
print(CONFIG, RUN_ID, OVERRIDES)

## 3. Corpus (once)

Builds `data/corpus.jsonl` on Drive from a Hugging Face dataset. Check the text column name of the dataset first. Decontamination against benchmark test sets is a separate step and is not done here.

In [ ]:
CORPUS = f'{DRIVE_ROOT}/data/corpus.jsonl'
os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)
if not os.path.exists(CORPUS):
    !python -m turkish_distill.prepare_corpus --dataset ytu-ce-cosmos/Cosmos-Turkish-Corpus-v1.0 --text-column text --max-samples 1000000 --out {CORPUS}
else:
    print('corpus already exists:', CORPUS)

## 4. Teacher embeddings (cached on Drive, resumable)

Needs a GPU with enough memory for the largest teacher (Qwen3-8B in bf16 is about 16 GB of weights).

In [ ]:
!python -m turkish_distill.encode_teachers --config {CONFIG} {override_args}

## 5. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/bitirme/runs

## 6. Train (resumes automatically if RUN_ID points to an existing run)

In [ ]:
run_id_arg = f'--run-id {RUN_ID}' if RUN_ID else ''
!python -m turkish_distill.train --config {CONFIG} {run_id_arg} {override_args}

## 7. Inspect metrics

In [ ]:
import glob, pandas as pd
from turkish_distill.logging_utils import read_metrics

paths = sorted(glob.glob(f'{DRIVE_ROOT}/runs/*/*/metrics.jsonl'))
print(*paths[-5:], sep='\n')
df = pd.DataFrame(read_metrics(paths[-1]))
df.filter(regex='^(step|diag/)').dropna(how='all', subset=[c for c in df.columns if c.startswith('diag/')]).tail(10)